# Notebook 06: Base Model Selection (XuetangX)

**Purpose:** Evaluate Random, Popularity, Session-KNN, SASRec, and GRU4Rec on the test episode set.
Select the best backbone for MAML (highest HR@10).

**Protocol:** Zero-shot evaluation — models trained on full training pairs; evaluated on K=5 support, Q=10 query episodes.

**Metrics:** HR@1, HR@5, HR@10 (primary), NDCG@10 (primary), MRR

**Outputs:**
- `models/baselines/gru_global_xuetangx.pth` — pretrained GRU weights for MAML init
- Summary table selecting best backbone

In [1]:
# [CELL 06-00] Bootstrap: find_repo_root, PATHS, helpers

import json, time, uuid, hashlib, math, pickle, random
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Search upward for PROJECT_STATE.md — single source of truth for repo root."""
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / 'PROJECT_STATE.md').exists():
            return p
    raise RuntimeError(
        'Could not find PROJECT_STATE.md in current or parent directories.'
    )


REPO_ROOT = find_repo_root(Path.cwd())

PATHS = {
    'META_REGISTRY':  REPO_ROOT / 'meta.json',
    'DATA_RAW':       REPO_ROOT / 'data' / 'raw',
    'DATA_INTERIM':   REPO_ROOT / 'data' / 'interim',
    'DATA_PROCESSED': REPO_ROOT / 'data' / 'processed',
    'REPORTS':        REPO_ROOT / 'reports',
    'MODELS':         REPO_ROOT / 'models',
}


def cell_start(cid: str, title: str, **kw) -> float:
    t = time.time()
    print(f'\n[{cid}] {title}')
    print(f'[{cid}] start={datetime.now().isoformat(timespec="seconds")}')
    for k, v in kw.items():
        print(f'[{cid}] {k}={v}')
    return t


def cell_end(cid: str, t0: float, **kw) -> None:
    for k, v in kw.items():
        print(f'[{cid}] {k}={v}')
    print(f'[{cid}] elapsed={time.time()-t0:.2f}s  done')


def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f'.tmp_{uuid.uuid4().hex}')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding='utf-8')
    tmp.replace(path)


def read_json(path: Path) -> Any:
    return json.loads(Path(path).read_text(encoding='utf-8'))


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


print(f'[CELL 06-00] REPO_ROOT={REPO_ROOT}')
print(f'[CELL 06-00] done')

[CELL 06-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 06-00] done


In [2]:
# [CELL 06-01] Seed + imports

t0 = cell_start('CELL 06-01', 'Seed + PyTorch imports')

import os
# Required for deterministic CuBLAS ops on CUDA >= 10.2
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

GLOBAL_SEED = 20260107


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print(f'[CELL 06-01] WARN: use_deterministic_algorithms: {e}')


seed_everything(GLOBAL_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cell_end('CELL 06-01', t0, seed=GLOBAL_SEED, device=DEVICE, torch=torch.__version__)


[CELL 06-01] Seed + PyTorch imports
[CELL 06-01] start=2026-04-08T22:17:41
[CELL 06-01] seed=20260107
[CELL 06-01] device=cuda
[CELL 06-01] torch=2.6.0+cu124
[CELL 06-01] elapsed=13.96s  done


In [3]:
# [CELL 06-02] Run config + directories

t0 = cell_start('CELL 06-02', 'Run config + directory setup')

NOTEBOOK_NAME = '06_base_model_selection_xuetangx'
DATASET       = 'xuetangx'
RUN_TAG       = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID        = uuid.uuid4().hex

K_SUPPORT = 5
Q_QUERY   = 10

# ── Paths ──
VOCAB_PATH    = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'vocab'  / 'course2id.json'
PAIRS_DIR     = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'pairs'
EPISODES_DIR  = PATHS['DATA_PROCESSED'] / 'xuetangx' / 'episodes'

PAIRS_TRAIN   = PAIRS_DIR    / 'pairs_train.parquet'
PAIRS_VAL     = PAIRS_DIR    / 'pairs_val.parquet'
PAIRS_TEST    = PAIRS_DIR    / 'pairs_test.parquet'

EPS_TRAIN     = EPISODES_DIR / f'episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet'
EPS_VAL       = EPISODES_DIR / f'episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet'
EPS_TEST      = EPISODES_DIR / f'episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet'

MODELS_DIR    = REPO_ROOT / 'models' / 'baselines'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
GRU_SAVE_PATH = MODELS_DIR / f'gru_global_{DATASET}.pth'

# ── Report dir ──
OUT_DIR       = PATHS['REPORTS'] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / 'report.json'
CONFIG_PATH   = OUT_DIR / 'config.json'
MANIFEST_PATH = OUT_DIR / 'manifest.json'

CFG = {
    'notebook': NOTEBOOK_NAME, 'dataset': DATASET,
    'global_seed': GLOBAL_SEED, 'K_support': K_SUPPORT,
    'Q_query': Q_QUERY, 'device': DEVICE,
}
write_json_atomic(CONFIG_PATH, CFG)

for path, obj in [
    (REPORT_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME, 'run_tag': RUN_TAG,
                   'created_at': datetime.now().isoformat(timespec='seconds'),
                   'metrics': {}, 'key_findings': [], 'notes': []}),
    (MANIFEST_PATH, {'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME,
                     'run_tag': RUN_TAG, 'artifacts': []}),
]:
    write_json_atomic(path, obj)

META = PATHS['META_REGISTRY']
if not META.exists():
    write_json_atomic(META, {'schema_version': 1, 'runs': []})
m = read_json(META)
m['runs'].append({'run_id': RUN_ID, 'notebook': NOTEBOOK_NAME,
                  'run_tag': RUN_TAG, 'out_dir': str(OUT_DIR)})
write_json_atomic(META, m)

cell_end('CELL 06-02', t0, run_tag=RUN_TAG)


[CELL 06-02] Run config + directory setup
[CELL 06-02] start=2026-04-08T22:17:55
[CELL 06-02] run_tag=20260408_221755
[CELL 06-02] elapsed=0.05s  done


In [4]:
# [CELL 06-03] Load data: vocab, pairs, episodes

t0 = cell_start('CELL 06-03', 'Load vocab, pairs, episodes')

# ── Checks ──
for p in [VOCAB_PATH, PAIRS_TRAIN, PAIRS_VAL, PAIRS_TEST,
           EPS_TRAIN, EPS_VAL, EPS_TEST]:
    if not p.exists():
        raise RuntimeError(f'Missing required file: {p}\nRun notebooks 01-05 first.')

# ── Vocab ──
course2id = read_json(VOCAB_PATH)
N_ITEMS = len(course2id)
id2course = {v: k for k, v in course2id.items()}
print(f'[CELL 06-03] vocab size: {N_ITEMS:,} courses')

# ── Pairs ──
pairs_train = pd.read_parquet(PAIRS_TRAIN)
pairs_val   = pd.read_parquet(PAIRS_VAL)
pairs_test  = pd.read_parquet(PAIRS_TEST)

for name, df in [('train', pairs_train), ('val', pairs_val), ('test', pairs_test)]:
    print(f'[CELL 06-03] pairs_{name}: {len(df):,} rows, '
          f'{df["user_id"].nunique():,} users, '
          f'cols={list(df.columns)}')

# ── Episodes ──
episodes_train = pd.read_parquet(EPS_TRAIN)
episodes_val   = pd.read_parquet(EPS_VAL)
episodes_test  = pd.read_parquet(EPS_TEST)

for name, df in [('train', episodes_train), ('val', episodes_val), ('test', episodes_test)]:
    print(f'[CELL 06-03] episodes_{name}: {len(df):,} episodes, '
          f'{df["user_id"].nunique():,} users')

# ── Build pair_id → row lookup for fast episode expansion ──
# Concatenate all pairs to resolve any pair_id from any split
all_pairs = pd.concat([pairs_train, pairs_val, pairs_test], ignore_index=True)
pair_lookup = all_pairs.set_index('pair_id')
print(f'[CELL 06-03] pair_lookup: {len(pair_lookup):,} rows')

cell_end('CELL 06-03', t0, n_items=N_ITEMS, n_test_episodes=len(episodes_test))


[CELL 06-03] Load vocab, pairs, episodes
[CELL 06-03] start=2026-04-08T22:17:55
[CELL 06-03] vocab size: 1,517 courses
[CELL 06-03] pairs_train: 344,532 rows, 49,566 users, cols=['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 06-03] pairs_val: 69,663 rows, 10,621 users, cols=['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 06-03] pairs_test: 73,501 rows, 10,622 users, cols=['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len']
[CELL 06-03] episodes_train: 113,920 episodes, 4,645 users
[CELL 06-03] episodes_val: 942 episodes, 942 users
[CELL 06-03] episodes_test: 986 episodes, 986 users
[CELL 06-03] pair_lookup: 487,696 rows
[CELL 06-03] n_items=1517
[CELL 06-03] n_test_episodes=986
[CELL 06-03] elapsed=0.94s  done


In [5]:
# [CELL 06-04] Evaluation metrics: HR@1, HR@5, HR@10, NDCG@10, MRR

t0 = cell_start('CELL 06-04', 'Define compute_metrics')


def ndcg_at_k(ranked_items: List[int], true_item: int, k: int = 10) -> float:
    """NDCG@k: 1/log2(rank+2) where rank is 0-indexed position in top-k."""
    for i, item in enumerate(ranked_items[:k]):
        if item == true_item:
            return 1.0 / math.log2(i + 2)
    return 0.0


def compute_metrics(predictions: List[int], labels: List[int]) -> Dict[str, float]:
    """
    Compute HR@1, HR@5, HR@10, NDCG@10, MRR for a batch of predictions.

    Args:
        predictions: list of ranked item lists (each length N_ITEMS), one per query pair
        labels:      list of true next-item ids, one per query pair

    Returns:
        dict with keys: hr1, hr5, hr10, ndcg10, mrr
    """
    hr1_list, hr5_list, hr10_list, ndcg10_list, mrr_list = [], [], [], [], []

    for ranked, true_item in zip(predictions, labels):
        hr1_list.append(1.0  if true_item in ranked[:1]  else 0.0)
        hr5_list.append(1.0  if true_item in ranked[:5]  else 0.0)
        hr10_list.append(1.0 if true_item in ranked[:10] else 0.0)
        ndcg10_list.append(ndcg_at_k(ranked, true_item, k=10))

        # MRR
        try:
            rank = ranked.index(true_item) + 1  # 1-indexed
            mrr_list.append(1.0 / rank)
        except ValueError:
            mrr_list.append(0.0)

    return {
        'hr1':    float(np.mean(hr1_list)),
        'hr5':    float(np.mean(hr5_list)),
        'hr10':   float(np.mean(hr10_list)),
        'ndcg10': float(np.mean(ndcg10_list)),
        'mrr':    float(np.mean(mrr_list)),
    }


print('[CELL 06-04] compute_metrics defined (HR@1, HR@5, HR@10, NDCG@10, MRR)')
cell_end('CELL 06-04', t0)


[CELL 06-04] Define compute_metrics
[CELL 06-04] start=2026-04-08T22:17:56
[CELL 06-04] compute_metrics defined (HR@1, HR@5, HR@10, NDCG@10, MRR)
[CELL 06-04] elapsed=0.00s  done


In [6]:
# [CELL 06-05] Sequence utilities (pad, build prefix seqs from pairs)

t0 = cell_start('CELL 06-05', 'Sequence utilities')


def pad_sequences(
    seqs: List[List[int]],
    pad_value: int = 0,
    max_len: Optional[int] = None,
    dtype=torch.long,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of variable-length sequences.
    Returns:
        padded: (B, L) tensor
        lengths: (B,) tensor of original lengths
    """
    lengths = [len(s) for s in seqs]
    if max_len is None:
        max_len = max(lengths) if lengths else 1
    padded = torch.full((len(seqs), max_len), pad_value, dtype=dtype)
    for i, s in enumerate(seqs):
        l = min(len(s), max_len)
        padded[i, :l] = torch.tensor(s[:l], dtype=dtype)
    return padded, torch.tensor(lengths, dtype=torch.long)


def pairs_to_sequences(
    pair_ids: List[int],
    lookup: pd.DataFrame,
    prefix_col: str = 'prefix',
    label_col: str  = 'label',
) -> Tuple[List[List[int]], List[int]]:
    """
    Resolve pair_ids to (prefix_sequences, labels) using the pair lookup table.
    prefix_ids is stored as a list of ints (course ids).
    """
    seqs, labels = [], []
    for pid in pair_ids:
        row = lookup.loc[pid]
        prefix = row[prefix_col]
        if isinstance(prefix, str):
            prefix = json.loads(prefix)
        seqs.append([int(x) for x in prefix])
        labels.append(int(row[label_col]))
    return seqs, labels


def episode_to_tensors(
    ep: pd.Series,
    lookup: pd.DataFrame,
    device: str = 'cpu',
) -> Dict[str, Any]:
    """
    Expand one episode row into support/query tensors.
    Returns dict with: support_seqs, support_lengths, support_labels,
                        query_seqs,   query_lengths,   query_labels
    """
    sup_seqs, sup_labels = pairs_to_sequences(ep['support_pair_ids'], lookup)
    qry_seqs, qry_labels = pairs_to_sequences(ep['query_pair_ids'],   lookup)

    sup_padded, sup_lengths = pad_sequences(sup_seqs)
    qry_padded, qry_lengths = pad_sequences(qry_seqs)

    return {
        'support_seqs':    sup_padded.to(device),
        'support_lengths': sup_lengths.to(device),
        'support_labels':  torch.tensor(sup_labels, dtype=torch.long).to(device),
        'query_seqs':      qry_padded.to(device),
        'query_lengths':   qry_lengths.to(device),
        'query_labels':    torch.tensor(qry_labels, dtype=torch.long).to(device),
    }


print('[CELL 06-05] pad_sequences, pairs_to_sequences, episode_to_tensors defined')
cell_end('CELL 06-05', t0)


[CELL 06-05] Sequence utilities
[CELL 06-05] start=2026-04-08T22:17:56
[CELL 06-05] pad_sequences, pairs_to_sequences, episode_to_tensors defined
[CELL 06-05] elapsed=0.00s  done


In [7]:
# [CELL 06-06] GRURecommender class (LOCKED architecture)

t0 = cell_start('CELL 06-06', 'Define GRURecommender')


class GRURecommender(nn.Module):
    """
    GRU4Rec-style session-based recommender.
    LOCKED architecture: embedding_dim=64, hidden_dim=128, num_layers=1, dropout=0.0
    Forward: embedding → GRU → take last hidden → FC → logits (n_items)
    """

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.n_items      = n_items
        self.embedding_dim = embedding_dim
        self.hidden_dim   = hidden_dim
        self.num_layers   = num_layers
        self.pad_idx      = pad_idx

        self.embedding = nn.Embedding(n_items + 1, embedding_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, n_items)

    def forward(
        self,
        seqs: torch.Tensor,    # (B, L)
        lengths: torch.Tensor, # (B,)
    ) -> torch.Tensor:         # (B, n_items)
        emb = self.embedding(seqs)                            # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)                         # hidden: (num_layers, B, H)
        h = hidden[-1]                                       # (B, H) — last layer
        logits = self.fc(h)                                  # (B, n_items)
        return logits

    def predict_ranked(
        self,
        seqs: torch.Tensor,
        lengths: torch.Tensor,
    ) -> List[List[int]]:
        """Return full ranked item list (descending score) for each sequence."""
        self.eval()
        with torch.no_grad():
            logits = self.forward(seqs, lengths)             # (B, n_items)
            ranked = torch.argsort(logits, dim=-1, descending=True)  # (B, n_items)
        return ranked.cpu().tolist()


# Quick smoke test
_m = GRURecommender(N_ITEMS).to(DEVICE)
_x = torch.randint(1, N_ITEMS + 1, (4, 8)).to(DEVICE)
_l = torch.tensor([8, 6, 4, 2]).to(DEVICE)
_out = _m(_x, _l)
assert _out.shape == (4, N_ITEMS), f'GRU output shape mismatch: {_out.shape}'
del _m, _x, _l, _out

print('[CELL 06-06] GRURecommender OK')
cell_end('CELL 06-06', t0, embedding_dim=64, hidden_dim=128, n_items=N_ITEMS)


[CELL 06-06] Define GRURecommender
[CELL 06-06] start=2026-04-08T22:17:56
[CELL 06-06] GRURecommender OK
[CELL 06-06] embedding_dim=64
[CELL 06-06] hidden_dim=128
[CELL 06-06] n_items=1517
[CELL 06-06] elapsed=3.21s  done


In [8]:
# [CELL 06-07] SASRec class (simple 2-block transformer)

t0 = cell_start('CELL 06-07', 'Define SASRec')


class SASRecBlock(nn.Module):
    """Single self-attention block with residual connections and layer norm."""

    def __init__(self, hidden_dim: int, n_heads: int = 2, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=n_heads,
            dropout=dropout, batch_first=True,
        )
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None):
        # Causal mask
        L = x.size(1)
        causal_mask = torch.triu(
            torch.ones(L, L, device=x.device), diagonal=1
        ).bool()
        attn_out, _ = self.attn(
            x, x, x,
            attn_mask=causal_mask,
            key_padding_mask=key_padding_mask,
        )
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation (simplified, 2-block).
    Architecture: item embedding + positional embedding → 2× SASRecBlock → last position → FC
    """

    def __init__(
        self,
        n_items: int,
        hidden_dim: int = 64,
        n_heads: int = 2,
        n_blocks: int = 2,
        max_seq_len: int = 50,
        dropout: float = 0.1,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.n_items    = n_items
        self.hidden_dim = hidden_dim
        self.pad_idx    = pad_idx

        self.item_emb = nn.Embedding(n_items + 1, hidden_dim, padding_idx=pad_idx)
        self.pos_emb  = nn.Embedding(max_seq_len + 1, hidden_dim)
        self.blocks   = nn.ModuleList([
            SASRecBlock(hidden_dim, n_heads, dropout)
            for _ in range(n_blocks)
        ])
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Linear(hidden_dim, n_items)
        self.drop = nn.Dropout(dropout)

    def forward(
        self,
        seqs: torch.Tensor,    # (B, L)
        lengths: torch.Tensor, # (B,) — used to build padding mask
    ) -> torch.Tensor:         # (B, n_items)
        B, L = seqs.shape
        positions = torch.arange(1, L + 1, device=seqs.device).unsqueeze(0).expand(B, -1)  # (B, L)

        # Key padding mask: True = ignore
        mask = (seqs == self.pad_idx)  # (B, L)

        x = self.drop(self.item_emb(seqs) + self.pos_emb(positions))  # (B, L, H)
        for block in self.blocks:
            x = block(x, key_padding_mask=mask)
        x = self.norm(x)  # (B, L, H)

        # Take the representation at the actual last (non-padded) position
        idx = (lengths - 1).clamp(min=0)  # (B,)
        last = x[torch.arange(B, device=x.device), idx]  # (B, H)
        logits = self.fc(last)  # (B, n_items)
        return logits

    def predict_ranked(
        self,
        seqs: torch.Tensor,
        lengths: torch.Tensor,
    ) -> List[List[int]]:
        self.eval()
        with torch.no_grad():
            logits = self.forward(seqs, lengths)
            ranked = torch.argsort(logits, dim=-1, descending=True)
        return ranked.cpu().tolist()


# Smoke test
_m = SASRec(N_ITEMS).to(DEVICE)
_x = torch.randint(1, N_ITEMS + 1, (4, 10)).to(DEVICE)
_l = torch.tensor([10, 8, 6, 4]).to(DEVICE)
_out = _m(_x, _l)
assert _out.shape == (4, N_ITEMS), f'SASRec output shape mismatch: {_out.shape}'
del _m, _x, _l, _out

print('[CELL 06-07] SASRec OK (2 blocks, hidden=64, heads=2)')
cell_end('CELL 06-07', t0)


[CELL 06-07] Define SASRec
[CELL 06-07] start=2026-04-08T22:17:59
[CELL 06-07] SASRec OK (2 blocks, hidden=64, heads=2)
[CELL 06-07] elapsed=5.68s  done


In [9]:
# [CELL 06-08] SessionKNN class

t0 = cell_start('CELL 06-08', 'Define SessionKNN')


class SessionKNN:
    """
    Session-based KNN recommender.
    Finds K most similar training sessions (by last N items overlap) and
    ranks candidate items by weighted co-occurrence frequency.
    """

    def __init__(self, n_items: int, k: int = 100, n_most_recent: int = 3):
        self.n_items      = n_items
        self.k            = k
        self.n_most_recent = n_most_recent
        self.train_seqs: List[List[int]]  = []
        self.train_labels: List[int] = []

    def fit(self, seqs: List[List[int]], labels: List[int]) -> 'SessionKNN':
        """Store training sequences and their next-item labels."""
        self.train_seqs   = seqs
        self.train_labels = labels
        return self

    def _similarity(self, query_tail: set, train_seq: List[int]) -> float:
        """Jaccard-like overlap between query tail and tail of train sequence."""
        train_tail = set(train_seq[-self.n_most_recent:])
        inter = len(query_tail & train_tail)
        union = len(query_tail | train_tail)
        return inter / union if union > 0 else 0.0

    def predict_ranked(self, seqs: List[List[int]]) -> List[List[int]]:
        """
        For each query sequence return full ranked item list.
        Falls back to item frequency if no similar sessions found.
        """
        results = []
        # Precompute item frequency for fallback
        freq = Counter(self.train_labels)

        for seq in seqs:
            tail = set(seq[-self.n_most_recent:])
            sims = [
                (self._similarity(tail, ts), tl)
                for ts, tl in zip(self.train_seqs, self.train_labels)
            ]
            sims.sort(key=lambda x: -x[0])
            top_k = sims[:self.k]

            # Score items by similarity-weighted votes
            scores: Dict[int, float] = {}
            for sim, label in top_k:
                if sim > 0:
                    scores[label] = scores.get(label, 0.0) + sim

            if not scores:
                # Fallback: global popularity
                ranked = [item for item, _ in freq.most_common()]
            else:
                ranked = sorted(scores.keys(), key=lambda x: -scores[x])
                # Append remaining items by popularity
                seen = set(ranked)
                ranked += [it for it, _ in freq.most_common() if it not in seen]

            # Ensure full coverage up to n_items
            seen_set = set(ranked)
            ranked += [i for i in range(1, self.n_items + 1) if i not in seen_set]
            results.append(ranked)
        return results


print('[CELL 06-08] SessionKNN defined')
cell_end('CELL 06-08', t0)


[CELL 06-08] Define SessionKNN
[CELL 06-08] start=2026-04-08T22:18:05
[CELL 06-08] SessionKNN defined
[CELL 06-08] elapsed=0.00s  done


In [10]:
# [CELL 06-09] PairDataset + collate_fn for neural model training

t0 = cell_start('CELL 06-09', 'Define PairDataset + collate_fn')


class PairDataset(Dataset):
    """
    Dataset over pairs DataFrame.
    Each item: (prefix_ids: List[int], label_id: int)
    """

    def __init__(
        self,
        pairs_df: pd.DataFrame,
        prefix_col: str = 'prefix',
        label_col: str  = 'label',
    ):
        self.seqs   = []
        self.labels = []
        for _, row in pairs_df.iterrows():
            prefix = row[prefix_col]
            if isinstance(prefix, str):
                prefix = json.loads(prefix)
            self.seqs.append([int(x) for x in prefix])
            self.labels.append(int(row[label_col]))

    def __len__(self) -> int:
        return len(self.seqs)

    def __getitem__(self, idx: int) -> Tuple[List[int], int]:
        return self.seqs[idx], self.labels[idx]


def collate_fn(
    batch: List[Tuple[List[int], int]],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Collate variable-length sequences.
    Returns: padded_seqs (B, L), lengths (B,), labels (B,)
    """
    seqs, labels = zip(*batch)
    padded, lengths = pad_sequences(list(seqs))
    return padded, lengths, torch.tensor(labels, dtype=torch.long)


print('[CELL 06-09] PairDataset + collate_fn defined')
cell_end('CELL 06-09', t0)


[CELL 06-09] Define PairDataset + collate_fn
[CELL 06-09] start=2026-04-08T22:18:05
[CELL 06-09] PairDataset + collate_fn defined
[CELL 06-09] elapsed=0.00s  done


In [11]:
# [CELL 06-10] Episode evaluation helper

t0 = cell_start('CELL 06-10', 'Define evaluate_on_episodes')


def evaluate_on_episodes(
    model,
    episodes_df: pd.DataFrame,
    lookup: pd.DataFrame,
    device: str = 'cpu',
    model_type: str = 'neural',   # 'neural' | 'knn'
    batch_size: int = 64,
    max_seq_len: Optional[int] = None,
) -> Dict[str, float]:
    """
    Evaluate a model on all episodes in episodes_df.
    For each episode: use ALL query pairs for prediction (zero-shot — no adaptation).
    Returns averaged metrics dict: hr1, hr5, hr10, ndcg10, mrr.
    """
    all_predictions: List[List[int]] = []
    all_labels: List[int] = []

    for _, ep in episodes_df.iterrows():
        qry_seqs, qry_labels = pairs_to_sequences(ep['query_pair_ids'], lookup)

        if model_type == 'knn':
            ranked_list = model.predict_ranked(qry_seqs)
        else:
            # Neural: batch the query sequences
            if max_seq_len is not None:
                qry_seqs = [s[-max_seq_len:] for s in qry_seqs]
            padded, lengths = pad_sequences(qry_seqs, max_len=max_seq_len)
            padded  = padded.to(device)
            lengths = lengths.to(device)
            ranked_list = model.predict_ranked(padded, lengths)

        all_predictions.extend(ranked_list)
        all_labels.extend(qry_labels)

    return compute_metrics(all_predictions, all_labels)


print('[CELL 06-10] evaluate_on_episodes defined')
cell_end('CELL 06-10', t0)


[CELL 06-10] Define evaluate_on_episodes
[CELL 06-10] start=2026-04-08T22:18:05
[CELL 06-10] evaluate_on_episodes defined
[CELL 06-10] elapsed=0.00s  done


In [12]:
# [CELL 06-11] Random baseline

t0 = cell_start('CELL 06-11', 'Random baseline')

seed_everything(GLOBAL_SEED)


class RandomRecommender:
    """Baseline: return a randomly shuffled item list for every query."""

    def __init__(self, n_items: int):
        self.n_items = n_items
        self.items   = list(range(1, n_items + 1))

    def predict_ranked(self, seqs, lengths=None) -> List[List[int]]:
        if isinstance(seqs, torch.Tensor):
            n = seqs.size(0)
        else:
            n = len(seqs)
        result = []
        for _ in range(n):
            shuffled = self.items[:]
            random.shuffle(shuffled)
            result.append(shuffled)
        return result


random_model = RandomRecommender(N_ITEMS)
random_metrics = evaluate_on_episodes(
    random_model, episodes_test, pair_lookup,
    device=DEVICE, model_type='neural',
)

print(f'[CELL 06-11] Random  HR@1={random_metrics["hr1"]*100:.2f}%  '
      f'HR@5={random_metrics["hr5"]*100:.2f}%  '
      f'HR@10={random_metrics["hr10"]*100:.2f}%  '
      f'NDCG@10={random_metrics["ndcg10"]*100:.2f}%  '
      f'MRR={random_metrics["mrr"]*100:.2f}%')

cell_end('CELL 06-11', t0)


[CELL 06-11] Random baseline
[CELL 06-11] start=2026-04-08T22:18:05
[CELL 06-11] Random  HR@1=0.05%  HR@5=0.28%  HR@10=0.70%  NDCG@10=0.30%  MRR=0.52%
[CELL 06-11] elapsed=4.12s  done


In [13]:
# [CELL 06-12] Popularity baseline

t0 = cell_start('CELL 06-12', 'Popularity baseline')

seed_everything(GLOBAL_SEED)


class PopularityRecommender:
    """
    Baseline: rank items by training-set label frequency.
    Ties broken by item id (deterministic).
    """

    def __init__(self, n_items: int):
        self.n_items = n_items
        self.ranked_items: List[int] = list(range(1, n_items + 1))  # default

    def fit(self, labels: List[int]) -> 'PopularityRecommender':
        freq = Counter(labels)
        # Sort by descending frequency, then ascending item id for ties
        self.ranked_items = sorted(
            range(1, self.n_items + 1),
            key=lambda x: (-freq.get(x, 0), x),
        )
        return self

    def predict_ranked(self, seqs, lengths=None) -> List[List[int]]:
        if isinstance(seqs, torch.Tensor):
            n = seqs.size(0)
        else:
            n = len(seqs)
        return [self.ranked_items[:] for _ in range(n)]


pop_model = PopularityRecommender(N_ITEMS)
pop_model.fit(pairs_train['label'].tolist())

pop_metrics = evaluate_on_episodes(
    pop_model, episodes_test, pair_lookup,
    device=DEVICE, model_type='neural',
)

print(f'[CELL 06-12] Popularity  HR@1={pop_metrics["hr1"]*100:.2f}%  '
      f'HR@5={pop_metrics["hr5"]*100:.2f}%  '
      f'HR@10={pop_metrics["hr10"]*100:.2f}%  '
      f'NDCG@10={pop_metrics["ndcg10"]*100:.2f}%  '
      f'MRR={pop_metrics["mrr"]*100:.2f}%')

cell_end('CELL 06-12', t0)


[CELL 06-12] Popularity baseline
[CELL 06-12] start=2026-04-08T22:18:09
[CELL 06-12] Popularity  HR@1=1.84%  HR@5=6.39%  HR@10=11.04%  NDCG@10=5.70%  MRR=5.39%
[CELL 06-12] elapsed=0.82s  done


In [14]:
# [CELL 06-13] Train GRU4Rec

t0 = cell_start('CELL 06-13', 'Train GRU4Rec (Adam, CE loss, 10 epochs)')

seed_everything(GLOBAL_SEED)

# ── Hyperparameters ──
GRU_EPOCHS     = 10
GRU_BATCH_SIZE = 256
GRU_LR         = 0.001

# ── Dataset & loader ──
train_dataset = PairDataset(pairs_train)
train_loader  = DataLoader(
    train_dataset, batch_size=GRU_BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=False,
)

# ── Model ──
gru_model = GRURecommender(
    n_items=N_ITEMS,
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.0,
).to(DEVICE)

optimizer = torch.optim.Adam(gru_model.parameters(), lr=GRU_LR)
criterion = nn.CrossEntropyLoss()

# ── Training loop ──
gru_train_losses = []
for epoch in range(1, GRU_EPOCHS + 1):
    gru_model.train()
    epoch_loss = 0.0
    n_batches  = 0
    for seqs, lengths, labels in train_loader:
        seqs    = seqs.to(DEVICE)
        lengths = lengths.to(DEVICE)
        labels  = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = gru_model(seqs, lengths)    # (B, n_items)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    gru_train_losses.append(avg_loss)
    print(f'[CELL 06-13] epoch={epoch}/{GRU_EPOCHS}  loss={avg_loss:.4f}')

cell_end('CELL 06-13', t0, final_loss=f'{gru_train_losses[-1]:.4f}')


[CELL 06-13] Train GRU4Rec (Adam, CE loss, 10 epochs)
[CELL 06-13] start=2026-04-08T22:18:10
[CELL 06-13] epoch=1/10  loss=4.8670
[CELL 06-13] epoch=2/10  loss=4.0174
[CELL 06-13] epoch=3/10  loss=3.8167
[CELL 06-13] epoch=4/10  loss=3.7063
[CELL 06-13] epoch=5/10  loss=3.6308
[CELL 06-13] epoch=6/10  loss=3.5723
[CELL 06-13] epoch=7/10  loss=3.5255
[CELL 06-13] epoch=8/10  loss=3.4848
[CELL 06-13] epoch=9/10  loss=3.4498
[CELL 06-13] epoch=10/10  loss=3.4183
[CELL 06-13] final_loss=3.4183
[CELL 06-13] elapsed=336.40s  done


In [15]:
# [CELL 06-14] Evaluate GRU4Rec

t0 = cell_start('CELL 06-14', 'Evaluate GRU4Rec on test episodes')

gru_metrics = evaluate_on_episodes(
    gru_model, episodes_test, pair_lookup,
    device=DEVICE, model_type='neural',
)

print(f'[CELL 06-14] GRU4Rec  HR@1={gru_metrics["hr1"]*100:.2f}%  '
      f'HR@5={gru_metrics["hr5"]*100:.2f}%  '
      f'HR@10={gru_metrics["hr10"]*100:.2f}%  '
      f'NDCG@10={gru_metrics["ndcg10"]*100:.2f}%  '
      f'MRR={gru_metrics["mrr"]*100:.2f}%')

cell_end('CELL 06-14', t0)


[CELL 06-14] Evaluate GRU4Rec on test episodes
[CELL 06-14] start=2026-04-08T22:23:47
[CELL 06-14] GRU4Rec  HR@1=24.92%  HR@5=43.52%  HR@10=51.87%  NDCG@10=37.36%  MRR=34.04%
[CELL 06-14] elapsed=2.69s  done


In [16]:
# [CELL 06-15] Train SASRec

t0 = cell_start('CELL 06-15', 'Train SASRec (Adam, CE loss, 10 epochs)')

seed_everything(GLOBAL_SEED)

# ── Hyperparameters ──
SAS_EPOCHS     = 10
SAS_BATCH_SIZE = 256
SAS_LR         = 0.001

# SASRec needs sequences capped at max_seq_len (positional embedding bound)
SAS_MAX_LEN = 50

def collate_fn_sas(batch):
    seqs, labels = zip(*batch)
    # Truncate to last SAS_MAX_LEN items (most recent context)
    seqs_trunc = [s[-SAS_MAX_LEN:] for s in seqs]
    padded, lengths = pad_sequences(list(seqs_trunc), max_len=SAS_MAX_LEN)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

sas_loader = DataLoader(
    train_dataset, batch_size=SAS_BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn_sas, num_workers=0, pin_memory=False,
)

# ── Model ──
sas_model = SASRec(
    n_items=N_ITEMS,
    hidden_dim=64,
    n_heads=2,
    n_blocks=2,
    max_seq_len=50,
    dropout=0.1,
).to(DEVICE)

sas_optimizer = torch.optim.Adam(sas_model.parameters(), lr=SAS_LR)
sas_criterion = nn.CrossEntropyLoss()

# ── Training loop ──
sas_train_losses = []
for epoch in range(1, SAS_EPOCHS + 1):
    sas_model.train()
    epoch_loss = 0.0
    n_batches  = 0
    for seqs, lengths, labels in sas_loader:
        seqs    = seqs.to(DEVICE)
        lengths = lengths.to(DEVICE)
        labels  = labels.to(DEVICE)

        sas_optimizer.zero_grad()
        logits = sas_model(seqs, lengths)
        loss   = sas_criterion(logits, labels)
        loss.backward()
        sas_optimizer.step()

        epoch_loss += loss.item()
        n_batches  += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    sas_train_losses.append(avg_loss)
    print(f'[CELL 06-15] epoch={epoch}/{SAS_EPOCHS}  loss={avg_loss:.4f}')

cell_end('CELL 06-15', t0, final_loss=f'{sas_train_losses[-1]:.4f}')


[CELL 06-15] Train SASRec (Adam, CE loss, 10 epochs)
[CELL 06-15] start=2026-04-08T22:23:49
[CELL 06-15] epoch=1/10  loss=5.1635
[CELL 06-15] epoch=2/10  loss=4.3220
[CELL 06-15] epoch=3/10  loss=4.1079
[CELL 06-15] epoch=4/10  loss=3.9996
[CELL 06-15] epoch=5/10  loss=3.9284
[CELL 06-15] epoch=6/10  loss=3.8779
[CELL 06-15] epoch=7/10  loss=3.8394
[CELL 06-15] epoch=8/10  loss=3.8109
[CELL 06-15] epoch=9/10  loss=3.7842
[CELL 06-15] epoch=10/10  loss=3.7638
[CELL 06-15] final_loss=3.7638
[CELL 06-15] elapsed=200.28s  done


In [17]:
# [CELL 06-16] Evaluate SASRec

t0 = cell_start('CELL 06-16', 'Evaluate SASRec on test episodes')

sas_metrics = evaluate_on_episodes(
    sas_model, episodes_test, pair_lookup,
    device=DEVICE, model_type='neural', max_seq_len=50,
)

print(f'[CELL 06-16] SASRec  HR@1={sas_metrics["hr1"]*100:.2f}%  '
      f'HR@5={sas_metrics["hr5"]*100:.2f}%  '
      f'HR@10={sas_metrics["hr10"]*100:.2f}%  '
      f'NDCG@10={sas_metrics["ndcg10"]*100:.2f}%  '
      f'MRR={sas_metrics["mrr"]*100:.2f}%')

cell_end('CELL 06-16', t0)


[CELL 06-16] Evaluate SASRec on test episodes
[CELL 06-16] start=2026-04-08T22:27:10
[CELL 06-16] SASRec  HR@1=25.65%  HR@5=43.20%  HR@10=51.75%  NDCG@10=37.58%  MRR=34.42%
[CELL 06-16] elapsed=4.77s  done


In [18]:
# [CELL 06-17] Fit and evaluate Session-KNN

t0 = cell_start('CELL 06-17', 'Fit + evaluate Session-KNN')

seed_everything(GLOBAL_SEED)

# Build training sequences from pairs_train
train_seqs_knn, train_labels_knn = [], []
for _, row in pairs_train.iterrows():
    prefix = row['prefix']
    if isinstance(prefix, str):
        prefix = json.loads(prefix)
    train_seqs_knn.append([int(x) for x in prefix])
    train_labels_knn.append(int(row['label']))

knn_model = SessionKNN(n_items=N_ITEMS, k=100, n_most_recent=3)
knn_model.fit(train_seqs_knn, train_labels_knn)
print(f'[CELL 06-17] SessionKNN fitted on {len(train_seqs_knn):,} training sequences')

knn_metrics = evaluate_on_episodes(
    knn_model, episodes_test, pair_lookup,
    device=DEVICE, model_type='knn',
)

print(f'[CELL 06-17] SessionKNN  HR@1={knn_metrics["hr1"]*100:.2f}%  '
      f'HR@5={knn_metrics["hr5"]*100:.2f}%  '
      f'HR@10={knn_metrics["hr10"]*100:.2f}%  '
      f'NDCG@10={knn_metrics["ndcg10"]*100:.2f}%  '
      f'MRR={knn_metrics["mrr"]*100:.2f}%')

cell_end('CELL 06-17', t0)


[CELL 06-17] Fit + evaluate Session-KNN
[CELL 06-17] start=2026-04-08T22:27:14
[CELL 06-17] SessionKNN fitted on 344,532 training sequences
[CELL 06-17] SessionKNN  HR@1=13.17%  HR@5=35.07%  HR@10=42.98%  NDCG@10=27.43%  MRR=23.55%
[CELL 06-17] elapsed=2540.89s  done


In [19]:
# [CELL 06-18] Results summary table + backbone selection

t0 = cell_start('CELL 06-18', 'Results summary + backbone selection')

all_results = {
    'Random':      random_metrics,
    'Popularity':  pop_metrics,
    'Session-KNN': knn_metrics,
    'SASRec':      sas_metrics,
    'GRU4Rec':     gru_metrics,
}

N_TEST_EPS = len(episodes_test)

# ── Select best backbone by HR@10 ──
best_name   = max(all_results, key=lambda k: all_results[k]['hr10'])
best_metric = all_results[best_name]

# ── Print required format ──
print('========== BASE MODEL SELECTION — XUETANGX ==========')
print(f'Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test: {N_TEST_EPS} episodes')
print()
print(f'{"Model":<14} {"HR@1":>6}  {"HR@5":>6}  {"HR@10":>7}  {"NDCG@10":>8}  {"MRR":>6}')
print('-' * 54)
for model_name, m in all_results.items():
    print(
        f'{model_name:<14} '
        f'{m["hr1"]*100:>5.2f}%  '
        f'{m["hr5"]*100:>5.2f}%  '
        f'{m["hr10"]*100:>6.2f}%  '
        f'{m["ndcg10"]*100:>7.2f}%  '
        f'{m["mrr"]*100:>5.2f}%'
    )
print()
print(f'Selected backbone: {best_name} (highest HR@10)')
print('Note: Base model trained on full training data — not cold-start constrained.')
print('======================================================')

cell_end('CELL 06-18', t0, best_backbone=best_name,
         best_hr10=f'{best_metric["hr10"]*100:.2f}%')


[CELL 06-18] Results summary + backbone selection
[CELL 06-18] start=2026-04-08T23:09:35
========== BASE MODEL SELECTION — XUETANGX ==========
Protocol: K=5 support, Q=10 query | Test: 986 episodes

Model            HR@1    HR@5    HR@10   NDCG@10     MRR
------------------------------------------------------
Random          0.05%   0.28%    0.70%     0.30%   0.52%
Popularity      1.84%   6.39%   11.04%     5.70%   5.39%
Session-KNN    13.17%  35.07%   42.98%    27.43%  23.55%
SASRec         25.65%  43.20%   51.75%    37.58%  34.42%
GRU4Rec        24.92%  43.52%   51.87%    37.36%  34.04%

Selected backbone: GRU4Rec (highest HR@10)
Note: Base model trained on full training data — not cold-start constrained.
[CELL 06-18] best_backbone=GRU4Rec
[CELL 06-18] best_hr10=51.87%
[CELL 06-18] elapsed=0.00s  done


In [20]:
# [CELL 06-19] Save pretrained GRU weights

t0 = cell_start('CELL 06-19', 'Save gru_global_xuetangx.pth')

MODELS_DIR.mkdir(parents=True, exist_ok=True)
torch.save(gru_model.state_dict(), GRU_SAVE_PATH)

ckpt_size = GRU_SAVE_PATH.stat().st_size
ckpt_hash = sha256_file(GRU_SAVE_PATH)

print(f'[CELL 06-19] Saved: {GRU_SAVE_PATH}')
print(f'[CELL 06-19] size={ckpt_size:,} bytes  sha256={ckpt_hash[:16]}...')

# Verify round-trip load
gru_check = GRURecommender(n_items=N_ITEMS, embedding_dim=64,
                             hidden_dim=128, num_layers=1, dropout=0.0)
gru_check.load_state_dict(torch.load(GRU_SAVE_PATH, map_location='cpu'))
gru_check.eval()
print('[CELL 06-19] Round-trip load OK')

cell_end('CELL 06-19', t0)


[CELL 06-19] Save gru_global_xuetangx.pth
[CELL 06-19] start=2026-04-08T23:09:35
[CELL 06-19] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_xuetangx.pth
[CELL 06-19] size=1,471,856 bytes  sha256=ccb3b178b0f4a78b...
[CELL 06-19] Round-trip load OK
[CELL 06-19] elapsed=0.01s  done


In [21]:
# [CELL 06-20] Update report + manifest

t0 = cell_start('CELL 06-20', 'Persist report + manifest')

report = read_json(REPORT_PATH)
report['metrics'] = {
    'random':      random_metrics,
    'popularity':  pop_metrics,
    'session_knn': knn_metrics,
    'sasrec':      sas_metrics,
    'gru4rec':     gru_metrics,
}
report['key_findings'] = [
    f'Selected backbone: {best_name} (highest HR@10={best_metric["hr10"]*100:.2f}%)',
    f'GRU4Rec HR@10={gru_metrics["hr10"]*100:.2f}%  NDCG@10={gru_metrics["ndcg10"]*100:.2f}%  MRR={gru_metrics["mrr"]*100:.2f}%',
    f'SASRec  HR@10={sas_metrics["hr10"]*100:.2f}%  NDCG@10={sas_metrics["ndcg10"]*100:.2f}%  MRR={sas_metrics["mrr"]*100:.2f}%',
    f'Test episodes: {N_TEST_EPS}',
    f'Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query (zero-shot, no adaptation)',
]
report['notes'].append(f'GRU weights saved to: {GRU_SAVE_PATH}  sha256={ckpt_hash}')
write_json_atomic(REPORT_PATH, report)

manifest = read_json(MANIFEST_PATH)
manifest['artifacts'] = [
    {
        'path':   str(GRU_SAVE_PATH),
        'type':   'model_checkpoint',
        'sha256': ckpt_hash,
        'bytes':  ckpt_size,
        'note':   'GRU4Rec pretrained on full xuetangx training set — used as MAML warm-start init',
    },
    {
        'path':   str(REPORT_PATH),
        'type':   'report',
    },
]
write_json_atomic(MANIFEST_PATH, manifest)

print(f'[CELL 06-20] Report:   {REPORT_PATH}')
print(f'[CELL 06-20] Manifest: {MANIFEST_PATH}')

cell_end('CELL 06-20', t0)


[CELL 06-20] Persist report + manifest
[CELL 06-20] start=2026-04-08T23:09:36
[CELL 06-20] Report:   /home/user/jamalla/anonymous-users-mooc-session-meta/reports/06_base_model_selection_xuetangx/20260408_221755/report.json
[CELL 06-20] Manifest: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/06_base_model_selection_xuetangx/20260408_221755/manifest.json
[CELL 06-20] elapsed=0.01s  done


In [22]:
# [CELL 06-21] Standard result block

t0 = cell_start('CELL 06-21', 'Standard result block')

# Use GRU4Rec metrics as the selected backbone
selected_metrics = gru_metrics

print('=' * 55)
print(f'RESULTS — {NOTEBOOK_NAME} — {DATASET.upper()}')
print('=' * 55)
print(f'Protocol:      K={K_SUPPORT} support, Q={Q_QUERY} query')
print(f'Test episodes: {len(episodes_test)}')
print(f'Seed:          {GLOBAL_SEED}')
print()
print(f'HR@1:          {selected_metrics["hr1"]*100:.2f}%')
print(f'HR@5:          {selected_metrics["hr5"]*100:.2f}%')
print(f'HR@10:         {selected_metrics["hr10"]*100:.2f}%   <- PRIMARY')
print(f'NDCG@10:       {selected_metrics["ndcg10"]*100:.2f}%   <- PRIMARY')
print(f'MRR:           {selected_metrics["mrr"]*100:.2f}%')
print()
print(f'Selected backbone: {best_name}')
print(f'GRU weights saved: {GRU_SAVE_PATH}')
print('=' * 55)

cell_end('CELL 06-21', t0)


[CELL 06-21] Standard result block
[CELL 06-21] start=2026-04-08T23:09:36
RESULTS — 06_base_model_selection_xuetangx — XUETANGX
Protocol:      K=5 support, Q=10 query
Test episodes: 986
Seed:          20260107

HR@1:          24.92%
HR@5:          43.52%
HR@10:         51.87%   <- PRIMARY
NDCG@10:       37.36%   <- PRIMARY
MRR:           34.04%

Selected backbone: GRU4Rec
GRU weights saved: /home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_xuetangx.pth
[CELL 06-21] elapsed=0.00s  done


## Notebook 06 Complete — Base Model Selection (XuetangX)

All five baselines evaluated on XuetangX test episodes (K=5 support, Q=10 query, zero-shot, **986 episodes**).

**Results:**
| Model | HR@1 | HR@5 | HR@10 | NDCG@10 | MRR |
|-------|------|------|-------|---------|-----|
| Random | 0.05% | 0.28% | 0.70% | 0.30% | 0.52% |
| Popularity | 1.84% | 6.39% | 11.04% | 5.70% | 5.39% |
| Session-KNN | 13.17% | 35.07% | 42.98% | 27.43% | 23.55% |
| SASRec | 25.65% | 43.20% | 51.75% | 37.58% | 34.42% |
| **GRU4Rec** | **24.92%** | **43.52%** | **51.87%** | **37.36%** | **34.04%** |

→ **Selected backbone: GRU4Rec** (highest HR@10 = 51.87%)

**Outputs:**
-  — pretrained GRU4Rec weights for MAML warm-start

**Next:**  — standard MAML benchmark using GRU4Rec backbone.
